# 英汉 / 汉英双向机器翻译（OPUS-MT）

使用本地预训练模型：`opus-mt-en-zh`（英→中）、`opus-mt-zh-en`（中→英）

技术框架：PyTorch + Hugging Face Transformers（MarianMT）

In [5]:
import sys
print("Python:", sys.version)
try:
    import torch, transformers, sentencepiece
    print("依赖齐全: torch", torch.__version__, "| transformers", transformers.__version__)
except ImportError as e:
    print("缺少依赖:", e)
    print("请运行以下两条命令安装后重启内核：")
    print("!" + sys.executable + " -m pip install torch --index-url https://download.pytorch.org/whl/cpu")
    print("!" + sys.executable + " -m pip install transformers sentencepiece sacremoses ipywidgets")

Python: 3.11.7 (tags/v3.11.7:fa7a6f2, Dec  4 2023, 19:24:49) [MSC v.1937 64 bit (AMD64)]


C:\Users\叶湘伦\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


依赖齐全: torch 2.14.0+cpu | transformers 5.17.0


In [6]:
import os, time
import torch
from transformers import MarianMTModel, MarianTokenizer

BASE      = r"C:\Users\叶湘伦\Desktop\2026-2027-1专业实习Transformers预训练模型\Bi_en_to_ch"
DIR_EN2ZH = os.path.join(BASE, "opus-mt-en-zh")   # 英→中
DIR_ZH2EN = os.path.join(BASE, "opus-mt-zh-en")   # 中→英

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备:", device)

_workdir = os.getcwd()

def load_model(path):
    """先切到模型目录再用相对路径加载，绕开 Windows 中文路径不被 SentencePiece 支持的问题"""
    os.chdir(path)
    tokenizer = MarianTokenizer.from_pretrained(".")
    model     = MarianMTModel.from_pretrained(".")
    model     = model.to(device).eval()
    n_params  = sum(p.numel() for p in model.parameters())
    os.chdir(_workdir)   # 恢复工作目录
    return tokenizer, model, n_params

print("加载 英→中 ...")
t0 = time.time()
tok_en2zh, model_en2zh, n1 = load_model(DIR_EN2ZH)
print(f"  完成，耗时 {time.time()-t0:.1f}s，参数量 {n1/1e6:.1f}M")

print("加载 中→英 ...")
t0 = time.time()
tok_zh2en, model_zh2en, n2 = load_model(DIR_ZH2EN)
print(f"  完成，耗时 {time.time()-t0:.1f}s，参数量 {n2/1e6:.1f}M")

使用设备: cpu
加载 英→中 ...


Loading weights: 100%|██████████| 258/258 [00:00<00:00, 34240.30it/s]


  完成，耗时 1.6s，参数量 77.9M
加载 中→英 ...


Loading weights: 100%|██████████| 258/258 [00:00<00:00, 24637.55it/s]


  完成，耗时 1.2s，参数量 77.9M


In [7]:
def translate(text, tokenizer, model, max_length=128, num_beams=4):
    inputs = tokenizer(text, return_tensors="pt", padding=True,
                       truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=max_length, num_beams=num_beams)
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

In [8]:
# 测试翻译
print("=== 英译中 ===")
for s in ["Transformers have revolutionized the field of natural language processing.",
          "The quick brown fox jumps over the lazy dog.",
          "I would like to book a ticket to Beijing tomorrow."]:
    t0 = time.time()
    print("EN:", s)
    print("ZH:", translate(s, tok_en2zh, model_en2zh))
    print(f"    耗时 {time.time()-t0:.2f}s\n")

print("=== 中译英 ===")
for s in ["Transformer 模型彻底改变了自然语言处理领域。",
          "人工智能是未来最重要的技术之一。",
          "请问去火车站怎么走？"]:
    t0 = time.time()
    print("ZH:", s)
    print("EN:", translate(s, tok_zh2en, model_zh2en, num_beams=6))
    print(f"    耗时 {time.time()-t0:.2f}s\n")

=== 英译中 ===
EN: Transformers have revolutionized the field of natural language processing.
ZH: 变异者使自然语言处理领域发生了革命性的变化。
    耗时 0.44s

EN: The quick brown fox jumps over the lazy dog.
ZH: 快速的棕色狐狸跳过懒狗
    耗时 0.35s

EN: I would like to book a ticket to Beijing tomorrow.
ZH: 明天我想订一张去北京的机票
    耗时 0.28s

=== 中译英 ===
ZH: Transformer 模型彻底改变了自然语言处理领域。
EN: The Transformer model radically changed the area of natural language processing.
    耗时 0.47s

ZH: 人工智能是未来最重要的技术之一。
EN: Artificial intelligence is one of the most important technologies in the future.
    耗时 0.50s

ZH: 请问去火车站怎么走？
EN: How do you get to the train station?
    耗时 0.34s



## 自己测试任意句子

在下方新单元格中运行：`translate("你的英文句子", tok_en2zh, model_en2zh)` 或 `translate("你的中文句子", tok_zh2en, model_zh2en, num_beams=6)`

In [9]:
translate("hello", tok_en2zh, model_en2zh)

'哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈哈'